# Chamada de variantes (*Variant Calling*)

Adaptado do [Data Carpentry — Wrangling Genomics](https://datacarpentry.github.io/wrangling-genomics/), [Episódio 4 (chamada de variantes)](https://datacarpentry.github.io/wrangling-genomics/04-variant_calling.html) e [Episódio 5 (automação)](https://datacarpentry.github.io/wrangling-genomics/05-automation.html).

As células deste notebook rodam em **Bash** (kernel `bash_kernel`), então os comandos são idênticos aos que você digitaria no terminal.

---

## Sobre os dados

Os dados desta aula vêm do **Experimento de Evolução de Longo Prazo** (*Long-Term Evolution Experiment*, **LTEE**), conduzido por **Richard Lenski** desde 1988. No LTEE, populações de *E. coli* são cultivadas continuamente em meio de cultivo limitado em glucose, mas suplementado com citrato, e, todo dia, uma fração é transferida para meio de cultivo "fresco" — por **dezenas de milhares de gerações**. Amostras são congeladas ao longo do caminho, criando um "registro fóssil" vivo que permite comparar a bactéria em diferentes momentos da evolução.

Vamos trabalhar com **três amostras de uma mesma população (Ara-3)**, sequenciadas em **gerações diferentes**:

| Amostra | Geração |
|---------|---------|
| `SRR2589044` | 5.000 |
| `SRR2584863` | 15.000 |
| `SRR2584866` | 50.000 |

Como todas descendem do mesmo ancestral, comparar seus genomas com a **referência REL606** (justamente esse linhagem ancestral) revela as **mutações que se acumularam** ao longo do tempo evolutivo. Nessas populações surgiram dois fenômenos marcantes: por volta da geração 31.000, uma população desenvolveu a capacidade de **usar citrato** — uma inovação envolvendo uma mutação rara em *E. coli*; e uma população tornou-se **hipermutadora**, acumulando mutações muito mais rápido.

> 📄 O experimento e a inovação do citrato são descritos em:
> Blount, Z. D., Borland, C. Z., & Lenski, R. E. (2008). Historical contingency and the evolution of a key innovation in an experimental population of *Escherichia coli*. *Proceedings of the National Academy of Sciences*, 105(23), 7899–7906. https://doi.org/10.1073/pnas.0803151105

---

## Objetivos

Ao final desta lição você será capaz de:

- **Inspecionar** um arquivo de alinhamento **BAM** (`samtools view`, `samtools flagstat`)
- **Indexar** um BAM (`samtools index`)
- **Chamar variantes** com `bcftools mpileup` e `bcftools call`
- **Filtrar** as variantes chamadas (`vcfutils.pl varFilter`)
- **Visualizar** as variantes no alinhamento, no terminal (`samtools tview`) e no **IGV-Web**
- **Automatizar** o pipeline para várias amostras com um laço `for`
- **Chamar variantes em conjunto** (*joint calling*), gerando um VCF **multi-amostra**

## Configuração inicial

Rode a célula abaixo **uma vez** para garantir que estamos na raiz do repositório (onde ficam as pastas `data/` e `results/`). A partir daí, todos os caminhos serão escritos como no terminal.

In [ ]:
# Garante que o diretório de trabalho é a raiz do repositório
# (onde ficam as pastas data/ e results/).
# Rode esta célula uma vez, no início. É seguro rodá-la mais de uma vez.
[ -d data ] || cd ..
pwd
echo "--- data/ (dados de entrada) ---"; ls data
echo "--- results/ (saídas) ---";        ls results

## Seção 0 — Preparação (já executada) ⚙️

> **Você não precisa rodar esta seção.** Os comandos abaixo já foram executados para preparar o material, e os resultados já estão no repositório. Estão aqui **apenas para reprodutibilidade** — para mostrar de onde vieram os arquivos BAM com que a aula começa.

Esta aula assume que você **já viu o mapeamento de *reads* (*read mapping*)**. Por isso, começamos a partir dos arquivos contendo os alinhamentos. As etapas de preparação foram:

1. **Download** do genoma de referência de *E. coli* e dos *reads* já filtrados (*trimmed*) das 3 amostras.
2. **Indexação** do genoma de referência para o mapeamento com o `bwa` (`bwa index`).
3. **Alinhamento** dos *reads* de cada amostra com `bwa mem` (com *read groups*) e **ordenação** por posição feita com `samtools sort`.

O resultado são três BAMs ordenados em `results/bam/` (`SRR2584863`, `SRR2584866`, `SRR2589044`). As Seções 1–6 usam o `SRR2584866`; as Seções 7–8 usam os três.

In [ ]:
# ⚙️ PREPARAÇÃO — já feita; NÃO é necessário rodar em aula.
# Mantido comentado apenas para reprodutibilidade (de onde vieram os BAMs iniciais).
#
# --- 1. Dados ---
# Genoma de referência: Escherichia coli B str. REL606 (NC_012967.1 / CP000819.1),
# 4.629.812 bp. Obtido do dataset oficial do Data Carpentry no figshare
# ("Data Carpentry Genomics beta 2.0", backup.tar.gz):
#   https://figshare.com/articles/dataset/Data_Carpentry_Genomics_beta_2_0/7726454
# -> data/ref_genome/ecoli_rel606.fasta
#
# ATENÇÃO:
#   (a) O link do NCBI que aparece na página do episódio
#       (GCA_000017985.1_ASM1798v1_genomic.fna.gz) baixa um arquivo corrompido.
#   (b) NÃO use o genoma de E. coli K-12 MG1655 (ASM584v2, 4.641.652 bp): os reads
#       são da linhagem B/REL606, e mapeá-los contra K-12 produz ~31.000 variantes
#       (divergência ENTRE linhagens) em vez das ~800 esperadas.
#
# Reads já filtrados (subconjunto do Data Carpentry, via figshare):
#   https://ndownloader.figshare.com/files/14418248  ->  data/trimmed_fastq_small/
#
# --- 2. Indexar a referência para o bwa ---
# bwa index data/ref_genome/ecoli_rel606.fasta
#
# --- 3. Alinhar as 3 amostras (bwa mem) e ordenar (samtools sort) ---
# A opção -R adiciona um "read group" com SM:<amostra>. É desse SM que o bcftools
# tira o NOME da amostra na chamada conjunta (Seção 8).
# mkdir -p results/bam
# for s in SRR2584863 SRR2584866 SRR2589044; do
#   bwa mem -R "@RG\tID:${s}\tSM:${s}\tPL:ILLUMINA" \
#     data/ref_genome/ecoli_rel606.fasta \
#     data/trimmed_fastq_small/${s}_1.trim.sub.fastq \
#     data/trimmed_fastq_small/${s}_2.trim.sub.fastq \
#   | samtools sort -o results/bam/${s}.aligned.sorted.bam -
# done

---

## Seção 1 — Inspecionar o alinhamento

Nossa aula começa aqui. Temos um alinhamento pronto e **ordenado por posição**:
`results/bam/SRR2584866.aligned.sorted.bam`.

O formato **BAM** é a versão **binária e comprimida** do formato **SAM** (texto). Por ser binário, não conseguimos lê-lo diretamente com `head` ou `cat` — usamos `samtools view` para traduzi-lo de volta para o formato SAM que é legível.

Vamos espiar as primeiras linhas de alinhamento:

In [ ]:
# Esse comando checa onde está o samtools? (confirma que estamos usando o do ambiente conda)
which samtools

# As primeiras linhas de alinhamento (samtools traduz o BAM binário de volta para o formato SAM)
samtools view results/bam/SRR2584866.aligned.sorted.bam | head

**Exemplo — as duas *strings* de um *read*.** Na saída acima, cada *read* traz (entre as colunas do SAM) a sua **sequência** e as suas **qualidades por base**:

```
SEQ  (DNA) : GTTGCACCGTTTGCTGCATGATATTGAAAAAAATATCACCAAATAAAAAACGCCTTAGTAAGTATTTTTCAGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGCTTCTGAACTG
QUAL       : C@@FFFFFHHHHHJJJIJIJJJJJJJJJJJJIJIJJJJIIJJIJJJJIJJJJJGHHFFDDFFCCDEFEEEEDDDDDDDDEEEEEEDCCDDDCDBDDDDDDCCDDDDDDCDCBDDDDDDDDDBDDDD@CDCCDDECCCDABCACDDECCDC
```

- **SEQ** — a sequência de DNA do *read* (coluna 10 do SAM).
- **QUAL** — a qualidade **por base**: uma letra por base, em código ASCII Phred+33 (coluna 11 do SAM). As duas *strings* têm o mesmo comprimento — uma qualidade para cada base.

> ⚠️ Não confundir **QUAL** (qualidade por base) com **MAPQ** (coluna 5): o MAPQ é um **único número por *read*** — a qualidade do **mapeamento** do *read* inteiro —, não uma *string*.

### Estatísticas gerais do alinhamento

O comando `samtools flagstat` dá um resumo rápido do alinhamento: total de *reads*, quantos foram mapeados, quantos formam pares corretamente alinhados, etc. É uma boa checagem **antes** de chamar variantes.

In [ ]:
# Resumo do alinhamento: total de reads, mapeados, pares corretos, etc.
samtools flagstat results/bam/SRR2584866.aligned.sorted.bam

---

## Seção 2 — Indexar o BAM

Assim como um índice no fim de um livro permite pular direto para uma página, o **índice de um BAM** (`.bai`) permite que ferramentas acessem rapidamente os *reads* de uma **região específica** do genoma, sem varrer o arquivo inteiro.

O BAM **precisa estar ordenado por posição** para ser indexado — o nosso já está (foi ordenado na preparação). O índice será necessário mais adiante, na etapa de **visualização** (`samtools tview`).

O comando gera um arquivo `.bai` ao lado do `.bam`:

In [ ]:
# Vamos olhar o que tem dentro da pasta antes de executar o comando abaixo
ls -lh results/bam/

In [ ]:
# Cria o índice (arquivo .bai ao lado do .bam)
samtools index results/bam/SRR2584866.aligned.sorted.bam

# Confirma que o índice foi criado
ls -lh results/bam/

---

## Seção 3 — Chamar variantes (*variant calling*)

Aqui está o núcleo da lição. A chamada de variantes com o `bcftools` tem **dois passos**:

**Passo 1 — `bcftools mpileup`:** percorre o genoma e, para cada posição, resume o que os *reads* alinhados mostram ali — profundidade de cobertura, bases observadas e suas qualidades. O resultado é um arquivo **BCF** (a versão binária do VCF) contendo as *verossimilhanças dos genótipos* em cada posição (Seção 9 para mais detalhes sobre a verossimilhança).

Passamos a **referência** com `-f`, para o `bcftools` saber qual é a base esperada em cada posição, e `-O b` para a saída sair em BCF binário. Primeiro criamos as pastas de saída:

In [ ]:
# Onde está o bcftools? (confirma que estamos usando o do ambiente conda)
which bcftools

# Pastas para os resultados (ignoradas pelo git — são saídas que você gera)
mkdir -p results/bcf results/vcf

# Passo 1: resume, posição a posição, o que os reads mostram (-> BCF)
bcftools mpileup -O b \
  -o results/bcf/SRR2584866_raw.bcf \
  -f data/ref_genome/ecoli_rel606.fasta \
  results/bam/SRR2584866.aligned.sorted.bam

**Passo 2 — `bcftools call`:** a partir das verossimilhanças do passo anterior, decide onde há de fato uma variante e escreve um arquivo **VCF**.

Três opções importantes:

- `--ploidy 1` — *E. coli* é **haploide** (um único cromossomo), então dizemos ao `bcftools` para **não** assumir genótipos diploides.
- `-m` — usa o modelo de chamada **multialélico** (recomendado; lida corretamente com posições que têm mais de um alelo alternativo).
- `-v` — reporta **apenas os sítios variantes** (*variant-only*), ignorando as posições idênticas à referência.

In [ ]:
# Passo 2: chama as variantes a partir do BCF (-> VCF)
bcftools call --ploidy 1 -m -v \
  -o results/vcf/SRR2584866_variants.vcf \
  results/bcf/SRR2584866_raw.bcf

### Espiando o VCF

O arquivo **VCF** (*Variant Call Format*) tem três partes: um **cabeçalho** (linhas iniciadas por `##`) que descreve como o arquivo foi gerado; uma **linha de nomes de colunas** (iniciada por um único `#`, com `#CHROM POS ID REF ALT ...`); e, em seguida, **uma linha por variante**.

Vamos ver as primeiras variantes, pulando o cabeçalho `##`:

In [ ]:
# Mostra a linha de colunas (#CHROM ...) e as primeiras variantes, sem o cabeçalho ##
grep -v "^##" results/vcf/SRR2584866_variants.vcf | head

### Entendendo as colunas do VCF

Cada linha de variante tem **8 colunas fixas**, seguidas de informações de genótipo. Tomando a primeira variante como exemplo:

```
NC_012967.1  1521  .  C  T  207.417  .  DP=9;...;MQ=60  GT:PL:AD  1:237,0:0,9
```

| # | Coluna | Valor no exemplo | Significado |
|---|--------|------------------|-------------|
| 1 | `CHROM`  | `NC_012967.1` | cromossomo / contig da referência |
| 2 | `POS`    | `1521` | posição na referência (contada a partir de 1) |
| 3 | `ID`     | `.` | identificador da variante (`.` = nenhum) |
| 4 | `REF`    | `C` | base na **referência** |
| 5 | `ALT`    | `T` | base **alternativa** encontrada nos *reads* |
| 6 | `QUAL`   | `207.417` | confiança na chamada, em escala Phred (**quanto maior, melhor**) |
| 7 | `FILTER` | `.` | status de filtragem (`.` = ainda não filtrado; `PASS` = passou) |
| 8 | `INFO`   | `DP=9;…;MQ=60` | anotações do sítio (ver abaixo) |

Depois das 8 colunas fixas vêm as de **genótipo**:

- `FORMAT` = `GT:PL:AD` — lista quais campos são reportados, na ordem;
- a última coluna (`1:237,0:0,9`) traz os **valores** desses campos para a nossa amostra.

**Alguns campos úteis:**

| Campo | Onde | Significado |
|-------|------|-------------|
| `DP`  | INFO / FORMAT | **profundidade**: nº de *reads* cobrindo a posição (aqui, 9) |
| `MQ`  | INFO | qualidade média de **mapeamento** dos *reads* (60 = ótima) |
| `GT`  | FORMAT | **genótipo**: `1` = alelo alternativo. É um único número porque *E. coli* é **haploide** (num diploide seria algo como `0/1` ou `1/1`) |
| `AD`  | FORMAT | *reads* que suportam cada alelo: `0,9` = 0 para `REF`, 9 para `ALT` |

> 💡 **Repare:** na posição 1521, `DP=9` e `AD=0,9` — **todos os 9 *reads* mostram o alelo alternativo**, nenhum mostra a referência. É uma variante bem sustentada. Guarde essa informação: vamos ver com mais detalhes como essa variante tem suporte estatístico na Seção 9.

---

## Seção 4 — Filtrar as variantes

As variantes que acabamos de chamar são **brutas**: entre elas há chamadas em que não confiamos muito — posições com pouquíssimos *reads*, com qualidade baixa, ou aglomeradas em torno de *indels* (onde o alinhamento é notoriamente instável).

O `vcfutils.pl` (que vem junto com o `bcftools`) traz o `varFilter`, que aplica um conjunto de **filtros padrão** e descarta essas chamadas menos confiáveis. Entre os critérios: profundidade mínima de *reads*, qualidade mínima de mapeamento, e remoção de variantes muito próximas de *indels*.

In [ ]:
# Onde está o vcfutils.pl? (repare: fica no mesmo diretório do bcftools —
# ele vem junto com o pacote)
which vcfutils.pl

# Aplica os filtros padrão do varFilter, gerando o VCF "final"
vcfutils.pl varFilter results/vcf/SRR2584866_variants.vcf \
  > results/vcf/SRR2584866_final_variants.vcf

### Quanto o filtro removeu?

Vamos comparar o número de variantes antes e depois:

In [ ]:
# Quantas variantes antes e depois do filtro?
echo "antes do filtro:  $(grep -vc '^#' results/vcf/SRR2584866_variants.vcf)"
echo "depois do filtro: $(grep -vc '^#' results/vcf/SRR2584866_final_variants.vcf)"

### Na prática: filtragem mais rigorosa

O `varFilter` é um filtro **conservador** — ele descarta o pior, não faz uma seleção rigorosa. Numa análise real, você normalmente aplicaria critérios explícitos e justificáveis, por exemplo com `bcftools filter`:

```bash
bcftools filter -i 'QUAL>=30 && DP>=10' results/vcf/SRR2584866_variants.vcf
```

Isso manteria apenas variantes com **qualidade** ≥ 30 e **profundidade** ≥ 10. Os limiares corretos dependem da cobertura, do organismo e da pergunta biológica — não existe um valor universal.

> 💡 **Pense a respeito:** por que filtrar por profundidade (`DP`) importa tanto? Uma variante observada em 2 *reads* e uma observada em 50 *reads* têm o mesmo grau de confiança? E se o organismo é diploide?

---

## Seção 5 — Visualizar as variantes

O `samtools tview` mostra os *reads* alinhados sobre a referência, direto no terminal. Normalmente ele é **interativo** (você navega com as setas do teclado), o que não funciona dentro de uma célula de notebook. Por isso usamos duas opções que o tornam **não-interativo**:

- `-d T` — imprime a saída como **texto puro**, em vez de abrir a interface navegável.
- `-p` — pula direto para uma **posição** específica do genoma.

Vamos olhar a variante da posição **1612** (`A` → `G`), uma das que sobreviveram à filtragem. Note que isto só funciona porque **indexamos o BAM** na Seção 2:

In [ ]:
# Visualiza o alinhamento na posição 1612, onde há uma variante A -> G.
#   -d T  : saída em texto puro (não-interativa), adequada para o notebook
#   -p    : pula direto para a posição indicada
samtools tview -d T -p NC_012967.1:1612 \
  results/bam/SRR2584866.aligned.sorted.bam \
  data/ref_genome/ecoli_rel606.fasta | head -20

### Como ler a saída do `tview`

A saída tem três partes, de cima para baixo:

1. Uma **régua** com as coordenadas do genoma.
2. A sequência da **referência**.
3. Os ***reads*** alinhados, um por linha.

Nos *reads*, a notação é:

| Símbolo | Significado |
|---------|-------------|
| `.` | base **igual** à referência, *read* na fita **direta** (*forward*) |
| `,` | base **igual** à referência, *read* na fita **reversa** |
| `A C G T` (maiúsculas) | base **diferente** da referência, fita direta |
| `a c g t` (minúsculas) | base **diferente** da referência, fita reversa |
| espaço em branco | região não coberta por aquele *read* |

Ou seja: **um mar de pontos e vírgulas com uma coluna de letras** é exatamente a assinatura visual de uma variante. Na posição 1612, a referência tem `A`, mas **todos os *reads* mostram `G`** — em ambas as fitas. Essa concordância entre fitas é um forte indício de que a variante é **real**, e não um artefato de sequenciamento.

> 🔍 **Exercício:** escolha outra posição da lista de variantes filtradas e visualize-a. Alguma parece menos convincente que a da posição 1612? O que a tornaria suspeita — poucos *reads*? Discordância entre as fitas? Bases variantes só em uma das pontas dos *reads*?

---

## Seção 6 — Extra: visualizar no IGV-Web 🔬

O `tview` é prático, mas não tem muitos recursos. O **IGV** (*Integrative Genomics Viewer*) é a ferramenta ideal para explorar dados genômicos visualmente.

Vamos usar o **IGV-Web**, que roda **inteiramente no navegador**: nada para instalar, em nenhum sistema operacional.

### 👉 [https://igv.org/app/](https://igv.org/app/)

### Passo 1 — Carregar o genoma de referência

O REL606 **não** está na lista de genomas prontos do IGV, então vamos carregá-lo manualmente.

No menu **`Genome`** → **`Local File...`**, selecione **os dois arquivos juntos**:

- `data/ref_genome/ecoli_rel606.fasta`
- `data/ref_genome/ecoli_rel606.fasta.fai`

> ⚠️ O `.fai` (o índice) é **obrigatório**. Sem ele o IGV não consegue navegar pelo FASTA. Selecione os dois arquivos na mesma caixa de diálogo.

### Passo 2 — Carregar as variantes

No menu **`Tracks`** → **`Local File...`**, selecione:

- `results/vcf/SRR2584866_final_variants.vcf`

Cada variante aparecerá como uma marca na trilha. Clicando numa delas, o IGV mostra os campos do VCF (posição, alelos `REF`/`ALT`, qualidade, profundidade).

### Passo 3 — Navegar até uma variante

Na caixa de busca, digite a mesma posição que vimos com o `tview`:

```
NC_012967.1:1612
```

> 💡 Se estiver rodando no **Binder**, os arquivos estão no servidor, não no seu computador. Baixe-os primeiro: no JupyterLab, botão direito no arquivo → **Download**. O VCF final tem só 110 KB e a referência 4,5 MB.

### Opcional — ver os *reads* também

Para reproduzir no IGV o que o `tview` mostrou, carregue também o alinhamento (`Tracks` → `Local File...`), selecionando **os dois arquivos juntos**:

- `results/bam/SRR2584866.aligned.sorted.bam`
- `results/bam/SRR2584866.aligned.sorted.bam.bai`

Navegando até `NC_012967.1:1612`, você verá a mesma coluna de bases discordantes da Seção 5 — agora colorida e navegável.

> 🔍 **Compare:** o que fica mais fácil de perceber no IGV do que no `tview`? E o que o `tview` tem de vantagem?

---

## Seção 7 — Automação: repetindo o pipeline para várias amostras

Até aqui trabalhamos com **uma** amostra (`SRR2584866`). Mas temos **três**: `SRR2584863`, `SRR2584866` e `SRR2589044`. Repetir os comandos das Seções 2–4 à mão para cada uma seria repetitivo e propenso a erros.

A solução é um **laço `for`**: escrevemos os passos **uma vez** e deixamos o computador repeti-los para cada amostra. Duas ideias do Bash deixam isso limpo:

- **variável** (`$sample`) — guarda o nome da amostra da vez;
- **`basename`** — extrai o nome da amostra a partir do caminho, removendo o sufixo. Ex.: `basename results/bam/SRR2584863.aligned.sorted.bam .aligned.sorted.bam` → `SRR2584863`.

O laço abaixo, para **cada** BAM ordenado: **indexa** (só se ainda não houver índice), e roda `mpileup` → `call` → `varFilter`, nomeando as saídas com `$sample`.

In [ ]:
# Laço sobre os 3 BAMs ordenados. Cada saída é nomeada pela amostra ($sample).
for bam in results/bam/*.aligned.sorted.bam; do
    sample=$(basename "$bam" .aligned.sorted.bam)
    echo ">>> Processando $sample"

    # Indexa APENAS se o índice .bai ainda não existir (evita retrabalho)
    if [ ! -f "${bam}.bai" ]; then
        samtools index "$bam"
    fi

    # Os mesmos passos das Seções 3 e 4, agora repetidos para cada amostra
    bcftools mpileup -O b -o results/bcf/${sample}_raw.bcf \
        -f data/ref_genome/ecoli_rel606.fasta "$bam"
    bcftools call --ploidy 1 -m -v \
        -o results/vcf/${sample}_variants.vcf results/bcf/${sample}_raw.bcf
    vcfutils.pl varFilter results/vcf/${sample}_variants.vcf \
        > results/vcf/${sample}_final_variants.vcf

    echo "    variantes filtradas: $(grep -vc '^#' results/vcf/${sample}_final_variants.vcf)"
done

### O que os números dizem

As contagens variam **muito** entre as amostras — e isso conta uma história. Lembre que cada amostra é de uma **geração** diferente da mesma população (Ara-3):

| Amostra | Geração | Variantes filtradas |
|---------|---------|---------------------|
| `SRR2589044` | 5.000 | ~10 |
| `SRR2584863` | 15.000 | ~25 |
| `SRR2584866` | 50.000 | ~775 |

O número de mutações **cresce com o tempo evolutivo** — esperado, já que mutações se acumulam geração após geração. Mas o salto da geração 15.000 para a 50.000 é grande demais para ser só isso: nesse intervalo, a linhagem evoluiu um **fenótipo hipermutador** e passou a acumular mutações muito mais rápido. **Os números refletem biologia real.**

> 💡 Os valores exatos podem variar um pouco: provavelmente o material preparado para o Data Carpentry utilizou versões diferentes dos programas utilizados aqui. Os números de SNPs no material são 10, 25 e 766.

---

## Seção 8 — Chamada conjunta: um VCF multi-amostra

Na Seção 7 geramos **um VCF por amostra**, cada um respondendo *"quais variantes ESTA amostra tem?"*. Mas muitas perguntas exigem **comparar** as amostras: *"neste sítio, qual é o genótipo de CADA uma?"*, *"esta mutação é compartilhada ou privada?"*.

Para isso fazemos a **chamada conjunta** (*joint calling*): passamos **todos os BAMs de uma vez** para o `bcftools mpileup`. O resultado é **um único VCF** com **uma coluna de genótipo por amostra** — uma matriz sítio × amostra.

> 🏷️ É aqui que os *read groups* fazem diferença: o rótulo `SM:` que definimos ao alinhar (na Seção 0 — Preparação) é de onde o `bcftools` tira os **nomes das amostras** que viram colunas. Sem eles, as colunas viriam nomeadas pelo caminho do arquivo BAM.

In [ ]:
# Chamada CONJUNTA: os 3 BAMs entram JUNTOS no mpileup -> um único VCF
bcftools mpileup -O b -o results/bcf/all_samples_raw.bcf \
  -f data/ref_genome/ecoli_rel606.fasta \
  results/bam/SRR2584863.aligned.sorted.bam \
  results/bam/SRR2584866.aligned.sorted.bam \
  results/bam/SRR2589044.aligned.sorted.bam

bcftools call --ploidy 1 -m -v \
  -o results/vcf/all_samples.vcf \
  results/bcf/all_samples_raw.bcf

In [ ]:
# Quais amostras estão no VCF conjunto?
echo "=== amostras ==="
bcftools query -l results/vcf/all_samples.vcf

# Matriz sítio x amostra: CHROM, POS, REF, ALT e a coluna de cada amostra.
# (Usamos awk para alinhar as colunas, sem depender do utilitário 'column'.)
echo; echo "=== primeiras variantes (SNPs) ==="
grep -v '^##' results/vcf/all_samples.vcf | grep -v INDEL | cut -f1,2,4,5,10- \
  | awk '{ for (i=1; i<=NF; i++) printf "%-14s", $i; print "" }' | head -8

### Lendo a matriz

Cada linha é um **sítio**; cada uma das últimas três colunas é uma **amostra**. O primeiro número do genótipo (`GT`, antes do `:`) diz o alelo naquela amostra:

- `0` = **igual à referência** naquele sítio;
- `1` = **alelo alternativo** (a variante).

Veja a posição **1521** (`C→T`): só o `SRR2584866` tem `1` — as outras duas têm `0`. Ou seja, essa mutação é **privada** do `SRR2584866`. Sítios com `1` em mais de uma amostra representam variação **compartilhada**.

**Esse é o ganho da chamada conjunta:** o VCF registra o genótipo de **todas** as amostras em **todos** os sítios variantes — inclusive as que são idênticas à referência ali. Na chamada por-amostra da Seção 7, o sítio 1521 simplesmente **não existiria** no VCF do `SRR2584863`, e não saberíamos se ali ele é igual à referência ou apenas "não foi chamado".

> 🔍 **Exercício:** encontre um sítio onde **duas** amostras têm o alelo alternativo (`1`) e uma tem a referência (`0`). Dica: dá para filtrar com `grep`/`awk`, ou carregar `results/vcf/all_samples.vcf` como *track* no IGV-Web (Seção 6) e comparar as amostras visualmente.

---

## Seção 9 — Extra: *under the hood* 🔬

> **Parte opcional / avançada.** Não é necessária para rodar o pipeline — é para quem quer entender **como**, exatamente, o `bcftools` decide uma variante. Ótima como aprofundamento depois da aula.

### Como o `bcftools` decide?

Ao longo da aula chamamos variantes sem abrir a "caixa-preta" do `bcftools`. Você pode ter pensado que ele simplesmente escolhe a base **mais frequente** entre os *reads* (uma "regra da maioria", *majority rule*). **Não é isso.** Ele usa um **modelo de verossimilhança**: cada *read* é **ponderado pela qualidade da sua base** (o Phred). Uma base de baixa qualidade contribui pouco; uma de alta qualidade, muito.

É para isso que serve o campo **`PL`**. Na posição 1521 (Seção 3) vimos `PL = 237,0`: são as verossimilhanças (em escala Phred) dos genótipos "referência" e "alternativo". O `0` marca o mais provável (alt); o `237` diz que "referência" é **astronomicamente** menos provável.

### A fórmula

Como *E. coli* é haploide, a verossimilhança de um alelo candidato $a$ num sítio é o produto, sobre todos os *reads*, da probabilidade de observar cada base $b_i$:

$$\mathcal{L}(a) = \prod_{i} P(b_i \mid a), \qquad P(b_i \mid a) = \begin{cases} 1 - e_i, & b_i = a \\ e_i/3, & b_i \neq a \end{cases}$$

em que $e_i = 10^{-Q_i/10}$ é a probabilidade de erro da base $i$ (do seu Phred $Q_i$). O `bcftools` faz essa conta para cada alelo possível e reporta o resultado, em escala Phred, no campo `PL`. O modelo completo ([Li 2011](https://doi.org/10.1093/bioinformatics/btr509)) refina isso levando em conta dependências entre os *reads*.

### Entendendo $a$ e $b_i$

Dois personagens na fórmula:

- **$b_i$ — a base *observada*.** É o que o *read* $i$ mostra naquela posição. É **dado** (você lê do *pileup*).
- **$a$ — o alelo *candidato* (uma hipótese).** Um "chute" de qual seria a base **verdadeira** ali (`A`, `C`, `G` ou `T`). Para cada candidato, perguntamos: *"**se** a verdade fosse $a$, qual a probabilidade de eu ter observado exatamente esses *reads*?"* — isso é $\mathcal{L}(a)$.

A regra é **por *read***: comparamos o observado ($b_i$) com a hipótese ($a$):

| Situação | Significado | Probabilidade |
|----------|-------------|---------------|
| $b_i = a$ | o *read* **concorda** com a hipótese (base lida corretamente) | $1 - e_i$ |
| $b_i \neq a$ | o *read* **discorda** → só possível com **erro**, caindo justo nessa base (1 das 3 outras) | $e_i/3$ |

### Exemplo de brinquedo: 3 *reads* mostrando `T`, `T`, `C`

Antes dos dados reais, um exemplo pequeno para ver a mecânica. Suponha **3 *reads*** cobrindo um sítio, todos com qualidade Phred 30 ($e = 0{,}001$), mostrando **`T`, `T`, `C`**. Testamos as duas hipóteses plausíveis:

$$\mathcal{L}(T) = \underbrace{(1-e)}_{\text{read 1 = T ✓}}\cdot\underbrace{(1-e)}_{\text{read 2 = T ✓}}\cdot\underbrace{(e/3)}_{\text{read 3 = C ✗}} \approx 3{,}3\times10^{-4}$$

$$\mathcal{L}(C) = \underbrace{(e/3)}_{\text{read 1 = T ✗}}\cdot\underbrace{(e/3)}_{\text{read 2 = T ✗}}\cdot\underbrace{(1-e)}_{\text{read 3 = C ✓}} \approx 1{,}1\times10^{-7}$$

`T` vence por **~3000×** — porque a maioria (ponderada por qualidade!) o apoia. Repare que **os dois termos** ($1-e$ e $e/3$) aparecem em **cada** hipótese; o que muda é *quais reads concordam* com ela. E não é um voto simples: se aquele único `C` tivesse qualidade altíssima e os `T` fossem ruins, a conta poderia virar.

### Duas perguntas que surgem naturalmente

**Por que não calcular os 4 alelos?** Em teoria, calcularíamos $\mathcal{L}(A), \mathcal{L}(C), \mathcal{L}(G), \mathcal{L}(T)$ e pegaríamos o maior. Na prática, alelos que **nenhum *read* apoia** exigiriam que *todos* os *reads* fossem erro — probabilidade ínfima, perdem com certeza. Por eficiência, o `bcftools` só avalia a sério o **alelo da referência + os alelos observados**. Por isso o `PL` costuma trazer poucos números.

**A referência entra no cálculo?** **Não** — repare que a fórmula de $\mathcal{L}(a)$ nunca usa a base da referência; ela só compara cada *read* ($b_i$) com a hipótese ($a$). A referência entra **depois**, em outra etapa:

| Etapa | Ferramenta | O que faz |
|-------|-----------|-----------|
| 1. Verossimilhanças | `bcftools mpileup` | calcula $\mathcal{L}(a)$ de cada candidato (o `PL`). **Nenhum genótipo escolhido ainda.** |
| 2. Chamada | `bcftools call` | escolhe o genótipo mais provável (com um **viés a favor da referência**, pois variantes são raras) e o **rotula** `0` (= referência) ou `1` (= alternativo). |

Ou seja: a verossimilhança (agnóstica à referência) vem **primeiro**; o `0`/`1` é atribuído **depois**, no `call`.

### Agora com dados reais: o *pileup* da posição 1521

O exemplo de brinquedo era inventado; vamos refazer a conta com os **dados de verdade** do nosso BAM. O `samtools mpileup` mostra a **coluna** de bases dos *reads* numa posição — a matéria-prima do cálculo. A saída traz: cromossomo, posição, base da referência, profundidade, as **bases** observadas nos *reads* e as **qualidades** (uma letra por base, em código ASCII Phred+33).

In [ ]:
# O "pileup" de uma única posição: base da referência, profundidade,
# as BASES dos reads (coluna 5) e as QUALIDADES (coluna 6, uma letra por base).
samtools mpileup -f data/ref_genome/ecoli_rel606.fasta \
  -r NC_012967.1:1521-1521 \
  results/bam/SRR2584866.aligned.sorted.bam

### Calculando o `PL` à mão

Na posição 1521 a referência é `C` e os **9 *reads* mostram `T`** (`TTtttTttT` — maiúsculo/minúsculo indica a fita). Decodificando as qualidades `<DGJFCDDF` (ASCII − 33) e convertendo em probabilidade de erro ($e_i = 10^{-Q_i/10}$):

| *read* | base | Phred $Q_i$ | $e_i$ |
|--------|------|-------------|-------|
| 1 | T | 27 | 0,0020 |
| 2 | T | 35 | 0,00032 |
| 3 | T | 38 | 0,00016 |
| 4 | T | 41 | 0,00008 |
| 5 | T | 37 | 0,00020 |
| 6 | T | 34 | 0,00040 |
| 7 | T | 35 | 0,00032 |
| 8 | T | 35 | 0,00032 |
| 9 | T | 37 | 0,00020 |

**Hipótese alt = `T`** (todos os *reads* concordam):
$$\mathcal{L}(T) = \prod_i (1 - e_i) \approx 0{,}996$$

**Hipótese ref = `C`** (todos os *reads* seriam erro):
$$\mathcal{L}(C) = \prod_i (e_i/3) \approx 6{,}4 \times 10^{-37}$$

Em escala Phred, cada hipótese vira um `PL` **relativo à melhor**:

$$\text{PL} = -10\,\log_{10}\!\left(\frac{\mathcal{L}}{\mathcal{L}_{\max}}\right),$$

onde $\mathcal{L}_{\max}$ é a **maior** verossimilhança entre as hipóteses — aqui, $\mathcal{L}(T)$. Assim, o genótipo **mais provável** sempre fica com **`PL = 0`** (ele é a *régua* da comparação), e os outros recebem um número **positivo** que mede *quão menos prováveis* são:

- alt = `T`: &nbsp; $\dfrac{\mathcal{L}(T)}{\mathcal{L}_{\max}} = 1 \;\Rightarrow\; \text{PL} = 0$
- ref = `C`: &nbsp; $\dfrac{\mathcal{L}(C)}{\mathcal{L}(T)} \approx 6{,}4\times10^{-37} \;\Rightarrow\; \text{PL} \approx 362$

O `bcftools` reportou `PL = 237,0`. **Mesma ordem de grandeza e mesma conclusão** — o alelo `T` vence de forma esmagadora — mas não idêntico: o `bcftools` é **mais conservador**. Ele aplica o **BAQ** (*Base Alignment Quality*), que reduz as qualidades perto de regiões de alinhamento incerto, e não trata os erros dos *reads* como perfeitamente independentes. Por isso chega a um número um pouco menor. **A intuição, porém, é exatamente a que calculamos à mão.**